In [1]:
import os
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np
learning_rate = 1e-4
keep_prob_rate = 0.7 #
max_epoch = 3
BATCH_SIZE = 50

DOWNLOAD_MNIST = False
if not(os.path.exists('./mnist/')) or not os.listdir('./mnist/'):
    # not mnist dir or mnist is empyt dir
    DOWNLOAD_MNIST = True


train_data = torchvision.datasets.MNIST(root='./mnist/',train=True, transform=torchvision.transforms.ToTensor(), download=DOWNLOAD_MNIST,)
train_loader = Data.DataLoader(dataset = train_data ,batch_size= BATCH_SIZE ,shuffle= True)

test_data = torchvision.datasets.MNIST(root = './mnist/', train = False)
test_x = torch.unsqueeze(test_data.data, dim=1).type(torch.FloatTensor)[:500] / 255.
test_y = test_data.targets[:500].numpy()

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d( 
                # MNIST 是灰度图，in_channels 为 1
                in_channels=1,  
                # 注释要求 32 out channels
                out_channels=32,
                # 注释要求 patch 7*7
                kernel_size=7,
                # 注释要求 stride 为 1
                stride=1,
                # 为了保持尺寸不变 (SAME)，padding = (kernel_size - 1) / 2 = (7-1)/2 = 3
                padding=3,
            ),
            nn.ReLU(),        # 激活函数
            nn.MaxPool2d(2),  # 池化后尺寸由 28x28 变为 14x14
        )
        self.conv2 = nn.Sequential( 
            # 填空：卷积 (32->64, kernel 5, stride 1, padding 2保持尺寸)
            nn.Conv2d(32, 64, kernel_size=5, stride=1, padding=2),
            # 填空：激活函数
            nn.ReLU(),
            # 填空：池化 (14x14 变为 7x7)
            nn.MaxPool2d(2),
        )
        
        # 经过两次 MaxPool2d(2)，图像尺寸从 28x28 -> 14x14 -> 7x7
        # 最终特征图大小为 7 * 7 * 64 (channels)
        self.out1 = nn.Linear( 7*7*64 , 1024 , bias= True)   

        self.dropout = nn.Dropout(keep_prob_rate)
        self.out2 = nn.Linear(1024, 10, bias=True)



    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # flatten the output of coonv2 to (batch_size ,32 * 7 * 7)    # ???
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        output = F.softmax(out2)
        return output


def test(cnn):
    global prediction
    y_pre = cnn(test_x)
    _,pre_index= torch.max(y_pre,1)
    pre_index= pre_index.view(-1)
    prediction = pre_index.data.numpy()
    correct  = np.sum(prediction == test_y)
    return correct / 500.0


def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate )
    loss_func = nn.CrossEntropyLoss()
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            x ,y= Variable(x_),Variable(y_)
            output = cnn(x)  
            loss = loss_func(output,y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            if step != 0 and step % 20 ==0:
                print("=" * 10,step,"="*5,"="*5, "test accuracy is ",test(cnn) ,"=" * 10 )

if __name__ == '__main__':
    cnn = CNN()
    train(cnn)




100%|██████████| 9.91M/9.91M [00:03<00:00, 3.18MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 167kB/s]
100%|██████████| 1.65M/1.65M [00:08<00:00, 185kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.8MB/s]
C:\Users\ROG\AppData\Local\Temp\ipykernel_29752\4168980460.py:72: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  output = F.softmax(out2)


========== 20 ===== ===== test accuracy is  0.1 ==========
========== 40 ===== ===== test accuracy is  0.29 ==========
========== 60 ===== ===== test accuracy is  0.404 ==========
========== 80 ===== ===== test accuracy is  0.502 ==========
========== 100 ===== ===== test accuracy is  0.564 ==========
========== 120 ===== ===== test accuracy is  0.62 ==========
========== 140 ===== ===== test accuracy is  0.642 ==========
========== 160 ===== ===== test accuracy is  0.666 ==========
========== 180 ===== ===== test accuracy is  0.718 ==========
========== 200 ===== ===== test accuracy is  0.74 ==========
========== 220 ===== ===== test accuracy is  0.788 ==========
========== 240 ===== ===== test accuracy is  0.79 ==========
========== 260 ===== ===== test accuracy is  0.81 ==========
========== 280 ===== ===== test accuracy is  0.824 ==========
========== 300 ===== ===== test accuracy is  0.86 ==========
========== 320 ===== ===== test accuracy is  0.876 ==========
========== 340 =====